In [ ]:
# Initialization and environment setup
%run ../../init.py
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from ipywidgets import interact, fixed
import ipywidgets as widgets
import helpers as hp

# Configure visual theme
sns.set_theme(style="whitegrid", palette="deep")

# Single-layer Networks: Regression
## Linear Basis Models, Maximum Likelihood, and Decision Theory

This lecture explores regression through the foundational framework of single-layer networks:
- **Linear Basis Function Models**: Extending linear representations via non-linear feature mappings
- **Maximum Likelihood & Normal Equations**: Deriving analytical least squares under Gaussian noise
- **Geometry of Least Squares**: Orthogonal projection onto the subspace spanned by basis vectors
- **Sequential Learning**: The LMS algorithm and the Widrow-Hoff Adaline rule
- **Regularized Least Squares**: Controlling complexity via weight decay (Ridge regression)
- **Decision Theory**: Optimal prediction under squared and Minkowski loss functions
- **The Bias–Variance Trade-off**: Decomposing expected generalization error

# Linear Basis Function Models
## Moving Beyond Raw Inputs

The simplest linear model expresses the target $y$ as a linear combination of raw input features:
$$y(\mathbf{x}, \mathbf{w}) = w_0 + w_1 x_1 + \dots + w_D x_D = \mathbf{w}^\mathrm{T} \mathbf{x}$$

However, this imposes a severe restriction: the model can only represent linear relationships.

To represent non-linear relationships while preserving linear optimization properties, we replace $\mathbf{x}$ with non-linear **basis functions** $\boldsymbol{\phi}(\mathbf{x})$:
$$y(\mathbf{x}, \mathbf{w}) = w_0 + \sum_{j=1}^{M-1} w_j \phi_j(\mathbf{x}) = \sum_{j=0}^{M-1} w_j \phi_j(\mathbf{x}) = \mathbf{w}^\mathrm{T} \boldsymbol{\phi}(\mathbf{x})$$

where $\phi_0(\mathbf{x}) \equiv 1$ serves as the bias/dummy basis function, $\mathbf{w} = (w_0, \dots, w_{M-1})^\mathrm{T}$, and $\boldsymbol{\phi}(\mathbf{x}) = (\phi_0(\mathbf{x}), \dots, \phi_{M-1}(\mathbf{x}))^\mathrm{T}$.

## Canonical Families of Basis Functions

The choice of basis functions $\phi_j(\mathbf{x})$ defines the feature representation:

1. **Polynomial Basis Functions**:
   $$\phi_j(x) = x^j$$
   Global functions; changes in one region affect predictions everywhere.

2. **Gaussian Radial Basis Functions (RBF)**:
   $$\phi_j(\mathbf{x}) = \exp\left(-\frac{\|\mathbf{x} - \boldsymbol{\mu}_j\|^2}{2s^2}\right)$$
   where $\boldsymbol{\mu}_j$ governs the center location and $s$ dictates spatial scale.

3. **Sigmoidal Basis Functions**:
   $$\phi_j(\mathbf{x}) = \sigma\left(\frac{\mathbf{x} - \boldsymbol{\mu}_j}{s}\right), \quad \sigma(a) = \frac{1}{1 + e^{-a}}$$

> 💡 **Key Takeaway:** The model is non-linear in input space $\mathbf{x}$, but strictly **linear in the parameter vector $\mathbf{w}$**. This preserves analytical tractability and convex optimization.

In [ ]:
# Demonstration: Polynomial vs Gaussian Basis Transformations
def polynomial_basis(x, degree=3):
    return torch.stack([x**i for i in range(degree + 1)], dim=1)

def gaussian_basis(x, centers, s=0.5):
    phi_0 = torch.ones_like(x).unsqueeze(1)
    phi_rbf = torch.stack([torch.exp(-((x - mu)**2) / (2 * s**2)) for mu in centers], dim=1)
    return torch.cat([phi_0, phi_rbf], dim=1)

# Concrete evaluation on 3 sample points
sample_x = torch.tensor([0.0, 0.5, 1.0])
phi_poly = polynomial_basis(sample_x, degree=2)
phi_gauss = gaussian_basis(sample_x, centers=[-0.5, 0.0, 0.5])

print("Sample Inputs x:\n", sample_x.numpy())
print("\nPolynomial Basis Matrix (Degree 2) [phi_0, phi_1, phi_2]:\n", phi_poly.numpy())
print("\nGaussian RBF Matrix (3 centers) [phi_0, mu_1, mu_2, mu_3]:\n", phi_gauss.numpy())

# Maximum Likelihood & Least Squares
## The Probabilistic Formulation

Assume target variable $t$ is given by deterministic function $y(\mathbf{x}, \mathbf{w})$ corrupted by additive Gaussian noise $\epsilon$:
$$t = y(\mathbf{x}, \mathbf{w}) + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \beta^{-1})$$
where $\beta$ denotes noise precision (inverse variance: $\beta = 1/\sigma^2$).

The conditional probability density over target $t$ given input $\mathbf{x}$ is:
$$p(t \mid \mathbf{x}, \mathbf{w}, \beta) = \mathcal{N}\left(t \mid y(\mathbf{x}, \mathbf{w}), \beta^{-1}\right) = \left(\frac{\beta}{2\pi}\right)^{1/2} \exp\left(-\frac{\beta}{2}\left(t - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})\right)^2\right)$$

For independently and identically distributed observations $\mathcal{D} = \{(\mathbf{x}_n, t_n)\}_{n=1}^N$, the likelihood is:
$$p(\mathbf{t} \mid \mathbf{X}, \mathbf{w}, \beta) = \prod_{n=1}^N \mathcal{N}\left(t_n \mid \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n), \beta^{-1}\right)$$

## Maximizing the Log-Likelihood

Taking the natural logarithm of the likelihood function:
$$\ln p(\mathbf{t} \mid \mathbf{w}, \beta) = \frac{N}{2}\ln \beta - \frac{N}{2}\ln(2\pi) - \beta E_\mathrm{D}(\mathbf{w})$$

where the sum-of-squares error function $E_\mathrm{D}(\mathbf{w})$ is defined as:
$$E_\mathrm{D}(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \left(t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n)\right)^2 = \frac{1}{2} \|\mathbf{t} - \boldsymbol{\Phi}\mathbf{w}\|^2$$

Here, $\boldsymbol{\Phi} \in \mathbb{R}^{N \times M}$ is the **design matrix**, whose rows are the basis function evaluations for each data point:
$$\boldsymbol{\Phi} = \begin{pmatrix} \phi_0(\mathbf{x}_1) & \phi_1(\mathbf{x}_1) & \dots & \phi_{M-1}(\mathbf{x}_1) \\ \phi_0(\mathbf{x}_2) & \phi_1(\mathbf{x}_2) & \dots & \phi_{M-1}(\mathbf{x}_2) \\ \vdots & \vdots & \ddots & \vdots \\ \phi_0(\mathbf{x}_N) & \phi_1(\mathbf{x}_N) & \dots & \phi_{M-1}(\mathbf{x}_N) \end{pmatrix}, \quad \mathbf{t} = \begin{pmatrix} t_1 \\ t_2 \\ \vdots \\ t_N \end{pmatrix}$$

> 💡 **Fundamental Equivalence:** Maximizing log-likelihood with respect to $\mathbf{w}$ under an additive Gaussian noise model is mathematically identical to minimizing sum-of-squares error.

## Deriving the Normal Equations

Setting the gradient of $E_\mathrm{D}(\mathbf{w})$ with respect to $\mathbf{w}$ to zero:
$$\nabla E_\mathrm{D}(\mathbf{w}) = \sum_{n=1}^N \left(\mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n) - t_n\right) \boldsymbol{\phi}(\mathbf{x}_n) = \boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}\mathbf{w} - \boldsymbol{\Phi}^\mathrm{T}\mathbf{t} = \mathbf{0}$$

Solving for $\mathbf{w}$ yields the canonical **Normal Equations**:
$$\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}\mathbf{w} = \boldsymbol{\Phi}^\mathrm{T}\mathbf{t}$$

Assuming $\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}$ is non-singular and invertible:
$$\mathbf{w}_\mathrm{ML} = \left(\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}\right)^{-1} \boldsymbol{\Phi}^\mathrm{T}\mathbf{t} = \boldsymbol{\Phi}^\dagger \mathbf{t}$$

where $\boldsymbol{\Phi}^\dagger = (\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^\mathrm{T}$ is the **Moore–Penrose pseudoinverse** of $\boldsymbol{\Phi}$.

Once $\mathbf{w}_\mathrm{ML}$ is obtained, the noise variance $\sigma^2 = \beta^{-1}$ is estimated analytically via maximum likelihood:
$$\frac{1}{\beta_\mathrm{ML}} = \frac{1}{N} \sum_{n=1}^N \left(t_n - \mathbf{w}_\mathrm{ML}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n)\right)^2 = \frac{2}{N} E_\mathrm{D}(\mathbf{w}_\mathrm{ML})$$

## Geometry of Least Squares

Least squares admits an illuminating geometric interpretation in $N$-dimensional sample space $\mathbb{R}^N$:

- The target vector $\mathbf{t} = (t_1, \dots, t_N)^\mathrm{T}$ is a point in $\mathbb{R}^N$.
- Each column of design matrix $\boldsymbol{\Phi}$ is an $N$-dimensional vector $\boldsymbol{\phi}_j = (\phi_j(\mathbf{x}_1), \dots, \phi_j(\mathbf{x}_N))^\mathrm{T}$.
- The columns span an $M$-dimensional linear subspace $\mathcal{S} \subset \mathbb{R}^N$.
- Any prediction vector $\mathbf{y} = \boldsymbol{\Phi}\mathbf{w}$ is a linear combination of these columns, lying strictly inside $\mathcal{S}$.

```
        t (Target in R^N)
       /|
      / |  residual: t - y (Orthogonal to S)
     /  |
    /___v
   0    y = Phi * w_ML (Orthogonal projection onto S)
   ============================= Subspace S (dim M)
```

> 📘 **Geometric Theorem:** The squared error $\|\mathbf{t} - \mathbf{y}\|^2$ is minimized when $\mathbf{y}$ is the **orthogonal projection** of $\mathbf{t}$ onto subspace $\mathcal{S}$. Consequently, the residual vector $\mathbf{t} - \mathbf{y}$ is strictly orthogonal to every basis vector:

$$\boldsymbol{\Phi}^\mathrm{T}(\mathbf{t} - \boldsymbol{\Phi}\mathbf{w}_\mathrm{ML}) = \mathbf{0}$$

# Sequential Learning & The LMS Algorithm
## Online Parameter Optimization

Inverting the $M \times M$ matrix $\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}$ requires $\mathcal{O}(M^3)$ operations, which becomes intractable for massive datasets or continuous real-time data streams.

**Sequential learning** updates weights incrementally upon observing each data point $(\mathbf{x}_n, t_n)$ using **Stochastic Gradient Descent (SGD)**:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \eta \nabla E_n(\mathbf{w})$$
where $\eta > 0$ is the learning rate, and $E_n(\mathbf{w}) = \frac{1}{2}(t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n)^2$.

Computing the gradient on sample $n$:
$$\nabla E_n(\mathbf{w}) = -(t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n)\boldsymbol{\phi}_n$$

This yields the celebrated **Least-Mean-Squares (LMS) Algorithm** (also known as the **Widrow-Hoff delta rule**):
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} + \eta \left(t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n\right) \boldsymbol{\phi}_n$$

<small>📖 *Source: [Widrow, B., & Hoff, M. E. (1960). Adaptive switching circuits. 1960 IRE WESCON Convention Record, 4, 96–104](https://doi.org/10.21236/ad0241531)*</small>

## Adaline: Adaptive Linear Neuron

In 1960, Bernard Widrow and Ted Hoff introduced **Adaline** (Adaptive Linear Neuron) as the first practical adaptive linear network:
- Unlike Rosenblatt's Perceptron (which updates on thresholded discrete outputs), Adaline computes errors on **continuous real-valued activations**: $y = \mathbf{w}^\mathrm{T}\mathbf{x}$.
- Adaline minimizes mean squared error via the LMS delta rule.

```
Inputs x_i ---> [ Sum: w^T x ] ---> continuous output y ---> Error: t - y (Updates weights)
                     |
                     v
             [ Threshold: sign(y) ] ---> discrete classification
```

In [ ]:
# Implementation of Adaline (Adaptive Linear Neuron) via Widrow-Hoff LMS
class AdaLine:
    def __init__(self, lr=0.01, epochs=200, tol=1e-4):
        self.lr, self.epochs, self.tol = lr, epochs, tol
        self.w, self.loss_history = None, []

    def fit(self, X, t):
        X_b = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
        N, D = X_b.shape
        self.w = torch.zeros(D, 1)

        for epoch in range(self.epochs):
            y = torch.matmul(X_b, self.w)
            error = t - y
            self.loss_history.append(torch.mean(error**2).item())
            # Widrow-Hoff batch gradient step: grad = -(2/N) * X^T (t - y)
            grad = - (2.0 / N) * torch.matmul(X_b.T, error)
            self.w -= self.lr * grad
            if epoch > 0 and abs(self.loss_history[-2] - self.loss_history[-1]) < self.tol:
                break
        return self

    def predict(self, X):
        X_b = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
        return torch.matmul(X_b, self.w)

In [ ]:
# Verification of Adaline on synthetic linear problem
synth_X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
synth_t = torch.tensor([[2.1], [3.9], [6.2], [7.9]])
model_adaline = AdaLine(lr=0.05, epochs=500).fit(synth_X, synth_t)

print("Trained Weights [w_0 (bias), w_1 (slope)]:\n", model_adaline.w.numpy().ravel())
print("Adaline Predictions:\n", model_adaline.predict(synth_X).numpy().ravel())
print(f"Final Mean Squared Error: {model_adaline.loss_history[-1]:.4f}")

# Empirical Case Study: Advertising Budget Regression
## Fitting Real-World Linear Models

We evaluate single-layer linear regression on the classic **Advertising dataset**:
- **Dataset**: 200 market regions recording product sales ($t$) against advertising budgets allocated to three distinct channels ($\mathbf{x}$):
  1. `TV`: Television budget (in thousands of dollars)
  2. `Radio`: Radio budget (in thousands of dollars)
  3. `Newspaper`: Newspaper budget (in thousands of dollars)
  4. `Sales`: Product sales (in thousands of units) — continuous regression target $t$.

We investigate:
1. **Univariate regression**: Predicting `Sales` strictly from `TV`.
2. **Multivariate regression**: Jointly predicting `Sales` from `TV`, `Radio`, and `Newspaper`.
3. **Statistical residuals diagnostics**: Total vs. Residual Sum of Squares (TSS, RSS, ESS, $R^2$).

In [ ]:
# Load Advertising dataset
adv_path = '../../shared/data/Advertising.csv'
adv = pd.read_csv(adv_path, dtype=float)
if 'Unnamed: 0' in adv.columns:
    adv = adv.drop(columns=['Unnamed: 0'])

print("Advertising Dataset Shape:", adv.shape)
print("Summary Statistics:\n", adv.describe().round(2))

# Visualize Univariate Relationship
plt.figure(figsize=(7, 4))
sns.scatterplot(data=adv, x="TV", y="Sales", color="royalblue", alpha=0.8, s=40)
plt.title("Product Sales vs TV Advertising Budget", fontsize=13)
plt.xlabel("TV Budget ($1,000s)")
plt.ylabel("Sales (1,000 units)")
plt.tight_layout()
plt.show()

In [ ]:
# Fit Univariate Normal Equations & Evaluate Residuals
X_tv = torch.tensor(adv[['TV']].values, dtype=torch.float32)
t_sales = torch.tensor(adv[['Sales']].values, dtype=torch.float32)

# Normal Equations solution: w_ML = (Phi^T Phi)^(-1) Phi^T t
Phi_tv = torch.cat([torch.ones(X_tv.shape[0], 1), X_tv], dim=1)
w_tv = torch.linalg.solve(torch.matmul(Phi_tv.T, Phi_tv), torch.matmul(Phi_tv.T, t_sales))

w0_tv, w1_tv = w_tv[0].item(), w_tv[1].item()
print(f"Normal Equations Analytical Solution:")
print(f"w_0 (Intercept): {w0_tv:.4f}")
print(f"w_1 (TV Slope) : {w1_tv:.4f}")
print(f"Model: Sales = {w0_tv:.4f} + {w1_tv:.4f} * TV")

# Compute predictions and evaluate residual breakdown
def predict_tv(X_in):
    if len(X_in.shape) == 1:
        X_in = X_in.unsqueeze(1)
    X_b = torch.cat([torch.ones(X_in.shape[0], 1), X_in], dim=1)
    return torch.matmul(X_b, w_tv)

hp.plot_residuals(predict_tv, X_tv, t_sales)

In [ ]:
# Multivariate Linear Regression: TV, Radio, Newspaper
X_all = torch.tensor(adv[['TV', 'Radio', 'Newspaper']].values, dtype=torch.float32)
Phi_all = torch.cat([torch.ones(X_all.shape[0], 1), X_all], dim=1)

# Normal equations: w_ML = (Phi^T Phi)^(-1) Phi^T t
w_all = torch.linalg.solve(torch.matmul(Phi_all.T, Phi_all), torch.matmul(Phi_all.T, t_sales))

features = ['Intercept (w_0)', 'TV (w_1)', 'Radio (w_2)', 'Newspaper (w_3)']
print("Multivariate Regression Parameters:")
for feat, weight in zip(features, w_all.squeeze().numpy()):
    print(f"  {feat:18s}: {weight:+.5f}")

# Compute goodness-of-fit R^2
y_pred_all = torch.matmul(Phi_all, w_all)
rss_all = torch.sum((t_sales - y_pred_all)**2).item()
tss_all = torch.sum((t_sales - torch.mean(t_sales))**2).item()
r2_all = 1.0 - (rss_all / tss_all)

print(f"\nMultivariate Goodness of Fit:")
print(f"  RSS = {rss_all:.2f}")
print(f"  TSS = {tss_all:.2f}")
print(f"  R^2 (Coefficient of Determination) = {r2_all:.4f} ({r2_all*100:.2f}% explained variance)")

# Regularized Least Squares
## Controlling Model Complexity via Weight Decay

When the number of basis functions $M$ is large relative to sample size $N$, or when features exhibit high multicollinearity, maximum likelihood overfits, leading to large parameter values with opposite signs.

To control complexity, we add an $L_2$ weight penalty (termed **Ridge regression** or **weight decay**):
$$E(\mathbf{w}) = E_\mathrm{D}(\mathbf{w}) + \frac{\lambda}{2} \|\mathbf{w}\|^2 = \frac{1}{2} \sum_{n=1}^N \left(t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n)\right)^2 + \frac{\lambda}{2} \mathbf{w}^\mathrm{T}\mathbf{w}$$
where $\lambda \ge 0$ is the regularization coefficient.

Setting the gradient to zero:
$$\nabla E(\mathbf{w}) = \boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}\mathbf{w} - \boldsymbol{\Phi}^\mathrm{T}\mathbf{t} + \lambda \mathbf{w} = \left(\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi} + \lambda \mathbf{I}\right)\mathbf{w} - \boldsymbol{\Phi}^\mathrm{T}\mathbf{t} = \mathbf{0}$$

Solving for $\mathbf{w}$ yields the **Regularized Normal Equations**:
$$\mathbf{w}_\mathrm{reg} = \left(\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi} + \lambda \mathbf{I}\right)^{-1} \boldsymbol{\Phi}^\mathrm{T}\mathbf{t}$$

> 💡 **Guaranteed Invertibility:** Even if $\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi}$ is singular or poorly conditioned, adding $\lambda \mathbf{I}$ with $\lambda > 0$ shifts all eigenvalues by $+\lambda$, making the matrix strictly positive-definite and safely invertible.

# Decision Theory for Regression
## Optimal Predictions under Uncertainty

In a probabilistic setting, the model predicts the full conditional distribution $p(t \mid \mathbf{x})$.
However, for practical decision-making, we must commit to a single point prediction $y(\mathbf{x})$.

**Decision theory** formalizes this choice by specifying a loss function $L(t, y(\mathbf{x}))$ and choosing $y(\mathbf{x})$ to minimize expected loss:
$$\mathbb{E}[L] = \iint L(t, y(\mathbf{x})) p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t$$

For the canonical **squared error loss** $L(t, y(\mathbf{x})) = (y(\mathbf{x}) - t)^2$:
$$\mathbb{E}[L] = \iint \left(y(\mathbf{x}) - t\right)^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t$$

## Deriving the Optimal Prediction under Squared Loss

Applying the calculus of variations and setting the functional derivative with respect to $y(\mathbf{x})$ to zero:
$$\frac{\delta \mathbb{E}[L]}{\delta y(\mathbf{x})} = 2 \int \left(y(\mathbf{x}) - t\right) p(\mathbf{x}, t) \mathrm{d}t = 0 \implies y(\mathbf{x}) \int p(\mathbf{x}, t) \mathrm{d}t = \int t p(\mathbf{x}, t) \mathrm{d}t$$

Dividing by $p(\mathbf{x})$ using the product rule $p(\mathbf{x}, t) = p(t \mid \mathbf{x}) p(\mathbf{x})$:
$$y^*(\mathbf{x}) = \int t p(t \mid \mathbf{x}) \mathrm{d}t = \mathbb{E}[t \mid \mathbf{x}]$$

> 📘 **Fundamental Result:** Under squared error loss, the optimal point prediction is the **conditional expectation** of the target variable conditioned on the input: $y^*(\mathbf{x}) = \mathbb{E}[t \mid \mathbf{x}]$.

## Generalized Loss: The Minkowski Metric

A natural generalization of squared loss is the **Minkowski loss**:
$$L_q(t, y(\mathbf{x})) = |y(\mathbf{x}) - t|^q, \quad q > 0$$

```
   q = 0.5 (Heavy outlier tolerance)     q = 1.0 (L1: Absolute Loss)      q = 2.0 (L2: Squared Loss)
            |   |                               \     /                          \       /
            |   |                                \   /                            \     /
           /     \                                \ /                              \___/
```

The choice of exponent $q$ determines which statistical property of $p(t \mid \mathbf{x})$ is predicted:
- **$q = 2$ (Squared Loss)**: Predicts the **conditional mean** $\mathbb{E}[t \mid \mathbf{x}]$. Sensitive to distant outliers.
- **$q = 1$ (Absolute Loss / MAE)**: Predicts the **conditional median** $\mathrm{median}(t \mid \mathbf{x})$. Robust to outliers.
- **$q \to 0$**: Predicts the **conditional mode** (most probable value).

# The Bias–Variance Trade-Off
## Decomposing Generalization Error

To evaluate how model complexity affects expected test error on unseen data, consider training a model $y(\mathbf{x}; \mathcal{D})$ on a dataset $\mathcal{D}$ drawn from true distribution $p(\mathbf{x}, t)$.

The expected squared loss on a specific point $\mathbf{x}$ across all possible training datasets $\mathcal{D}$ decomposes into three orthogonal components:
$$\mathbb{E}_{\mathcal{D}}\left[\left(y(\mathbf{x}; \mathcal{D}) - t\right)^2\right] = \left(\mathbb{E}_{\mathcal{D}}[y(\mathbf{x}; \mathcal{D})] - h(\mathbf{x})\right)^2 + \mathbb{E}_{\mathcal{D}}\left[\left(y(\mathbf{x}; \mathcal{D}) - \mathbb{E}_{\mathcal{D}}[y(\mathbf{x}; \mathcal{D})]\right)^2\right] + \sigma^2$$

where $h(\mathbf{x}) = \mathbb{E}[t \mid \mathbf{x}]$ is the true regression function, and $\sigma^2$ is the irreducible data noise variance.

Integrating over all inputs $\mathbf{x}$:
$$\text{Expected Test Loss} = (\text{Bias})^2 + \text{Variance} + \text{Irreducible Noise}$$

- **High Bias (Underfitting)**: Model is too rigid; fails to capture true underlying structure.
- **High Variance (Overfitting)**: Model is overly flexible; overly sensitive to idiosyncratic noise in training set $\mathcal{D}$.
- **Optimal Trade-off**: Balances bias and variance via regularization $\lambda$ or architecture selection.

In [ ]:
# Interactive Simulation Function: Bias-Variance Decomposition
def run_bias_variance_sim(n_datasets=50, n_samples=25, noise_std=0.2, degree=6):
    torch.manual_seed(42)
    x_test = torch.linspace(0, 1, 100)
    true_f = lambda x: torch.sin(2 * math.pi * x)
    y_test_true = true_f(x_test)

    lambdas = [1e-1, 1e-3, 1e-6]
    titles = [f"High Reg: lambda={l:.0e} (High Bias)" if l > 1e-2 
              else (f"Low Reg: lambda={l:.0e} (High Variance)" if l < 1e-4 
              else f"Optimal Reg: lambda={l:.0e}") for l in lambdas]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

    for ax, lam, title in zip(axes, lambdas, titles):
        preds = []
        for _ in range(n_datasets):
            x_train = torch.rand(n_samples)
            t_train = true_f(x_train) + torch.randn(n_samples) * noise_std

            Phi_train = torch.stack([x_train**p for p in range(degree + 1)], dim=1)
            Phi_test = torch.stack([x_test**p for p in range(degree + 1)], dim=1)

            reg_mat = lam * torch.eye(degree + 1)
            w_reg = torch.linalg.solve(Phi_train.T @ Phi_train + reg_mat, Phi_train.T @ t_train)
            y_pred = Phi_test @ w_reg
            preds.append(y_pred)
            ax.plot(x_test.numpy(), y_pred.numpy(), color='gray', alpha=0.15, lw=1)

        mean_pred = torch.stack(preds).mean(dim=0)
        ax.plot(x_test.numpy(), y_test_true.numpy(), 'g--', lw=2.5, label='Truth: sin(2pi x)')
        ax.plot(x_test.numpy(), mean_pred.numpy(), 'r-', lw=2.5, label='Mean Prediction E[y]')
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel("x")
        ax.legend(loc='lower left', fontsize=9)
        ax.set_ylim(-1.6, 1.6)

    plt.suptitle("Bias-Variance Decomposition across Regularization Strength", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive exploration of polynomial degree and noise level
interact(run_bias_variance_sim, 
         degree=widgets.IntSlider(min=1, max=8, step=1, value=6, description='Degree:'),
         noise_std=widgets.FloatSlider(min=0.05, max=0.4, step=0.05, value=0.2, description='Noise:'),
         n_samples=fixed(25), n_datasets=fixed(50));

# Summary: Single-layer Networks for Regression

In this lecture, we developed the theoretical foundation of linear regression:

1. **Linear Basis Models**:
   $$y(\mathbf{x}, \mathbf{w}) = \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})$$
   Enables non-linear feature representations while preserving convex linear optimization.

2. **Maximum Likelihood & Least Squares**:
   Assuming Gaussian noise $\epsilon \sim \mathcal{N}(0, \beta^{-1})$, maximum likelihood yields the analytical **Normal Equations**:
   $$\mathbf{w}_\mathrm{ML} = (\boldsymbol{\Phi}^\mathrm{T}\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^\mathrm{T}\mathbf{t}$$

3. **Geometry of Least Squares**:
   Predictions $\mathbf{y}$ are the orthogonal projection of target vector $\mathbf{t}$ onto the subspace $\mathcal{S}$ spanned by the columns of design matrix $\boldsymbol{\Phi}$.

4. **Sequential Learning (LMS / Adaline)**:
   Online updates via the Widrow-Hoff delta rule: $\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} + \eta (t_n - \mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n)\boldsymbol{\phi}_n$.

5. **Decision Theory & Bias-Variance**:
   Under squared loss, optimal prediction is conditional mean $\mathbb{E}[t \mid \mathbf{x}]$. Generalization error decomposes cleanly into $(\mathrm{bias})^2 + \mathrm{variance} + \sigma^2$.